# Harmonized ML classification (fills the one remaining number in the manuscript)

Run on the harmonized feature matrices saved by the fix notebook
(`features_ComBat_GSR_FIXED.csv`, `features_ComBat_noGSR_FIXED.csv`). This reproduces your
five-classifier comparison with the same leakage-controlled nested pipeline, on harmonized data.
Report the printed accuracy/AUC for both pipelines and I will place them in the manuscript
(Tables 7, 8, 11 and the abstract).

To regenerate the **figures** (whole/intra/inter boxplots, correlation heatmaps, H-correlates,
confusion/ROC/barplot), re-run your original plotting cells but point them at the harmonized
`features_ComBat_*_FIXED.csv` instead of the broken harmonized_df.


In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     cross_val_score, StratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, roc_curve)
try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

RESULTS = '/kaggle/working/results'
os.makedirs(RESULTS, exist_ok=True)

ASD_COLOR = '#DA005D'   # ASD
CTRL_COLOR = '#00473A'  # Control / TD


def run_ml(csv, label, seed=42):
    df = pd.read_csv(csv)
    meta = [c for c in ['Subject', 'Group', 'Center', 'IQ', 'FIQ'] if c in df.columns]
    feats = [c for c in df.columns if c not in meta]

    # NaNs are kept here on purpose: imputation now happens *inside* the CV
    # pipeline so the train folds never see test-set statistics (no leakage).
    X = df[feats].values.astype(float)
    y = (df['Group'] == 'ASD').astype(int).values

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed)

    models = {
        'GaussianNB': (GaussianNB(), {}),
        'SVM': (SVC(kernel='rbf', probability=True, random_state=seed),
                {'clf__C': [0.01, 0.1, 1, 10, 100],
                 'clf__gamma': ['scale', 'auto']}),
        'LogReg': (LogisticRegression(max_iter=5000, random_state=seed),
                   {'clf__C': [0.01, 0.1, 1, 10, 100]}),
        'KNN': (KNeighborsClassifier(),
                {'clf__n_neighbors': [3, 5, 7, 9, 11],
                 'clf__weights': ['uniform', 'distance']}),
    }
    if HAVE_XGB:
        models['XGBoost'] = (
            XGBClassifier(eval_metric='logloss', random_state=seed, verbosity=0),
            {'clf__max_depth': [2, 3, 4],
             'clf__n_estimators': [100, 200, 300],
             'clf__learning_rate': [0.05, 0.1]})

    print(f"\n==== {label}  (harmonized, n={len(df)}, "
          f"ASD={y.sum()}, TD={(1 - y).sum()}) ====")

    rows, fitted = [], {}
    for name, (est, grid) in models.items():
        pipe = Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc', StandardScaler()),
            ('clf', est),
        ])
        if grid:
            gs = GridSearchCV(pipe, grid, cv=cv, scoring='accuracy',
                              n_jobs=-1).fit(Xtr, ytr)
            mdl, cvs, bp = gs.best_estimator_, gs.best_score_, gs.best_params_
        else:
            cvs = cross_val_score(pipe, Xtr, ytr, cv=cv,
                                  scoring='accuracy').mean()
            mdl, bp = pipe.fit(Xtr, ytr), {}

        yp = mdl.predict(Xte)
        try:
            auc = roc_auc_score(yte, mdl.predict_proba(Xte)[:, 1])
        except Exception:
            auc = float('nan')
        acc = accuracy_score(yte, yp)
        bal = balanced_accuracy_score(yte, yp)
        f1 = f1_score(yte, yp)

        rows.append(dict(Model=name, CV_acc=cvs, Test_acc=acc,
                         Balanced_acc=bal, F1=f1, AUC=auc, best_params=str(bp)))
        fitted[name] = mdl
        print(f"  {name:11s} CV={cvs:.3f}  Test acc={acc:.3f}  "
              f"bal={bal:.3f}  F1={f1:.3f}  AUC={auc:.3f}")

    res = pd.DataFrame(rows)

    # Model selection by CROSS-VALIDATION accuracy, not test accuracy.
    # Picking the model that happens to score best on the held-out test set
    # biases that number upward; selecting on CV keeps the test set as an
    # honest, untouched estimate of generalization.
    best_idx = int(res['CV_acc'].idxmax())
    best_name = res.loc[best_idx, 'Model']

    # ---------------- per-plot PDF output ----------------
    def save(fig, name):
        path = os.path.join(RESULTS, f'{label}_{name}.pdf')
        fig.savefig(path, bbox_inches='tight')
        plt.close(fig)
        return path

    if True:

        # Plot 1 -- results table
        fig, ax = plt.subplots(figsize=(11, 6))
        ax.axis('off')
        ax.set_title(f'{label}: classifier comparison '
                     f'(n={len(df)}, ASD={y.sum()}, TD={(1 - y).sum()})',
                     fontsize=13, fontweight='bold', pad=20)
        show = res[['Model', 'CV_acc', 'Test_acc', 'Balanced_acc', 'F1', 'AUC']].copy()
        for c in ['CV_acc', 'Test_acc', 'Balanced_acc', 'F1', 'AUC']:
            show[c] = show[c].map(lambda v: f'{v:.3f}')
        tbl = ax.table(cellText=show.values, colLabels=show.columns,
                       loc='center', cellLoc='center')
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(10)
        tbl.scale(1, 1.6)
        for j in range(len(show.columns)):            # highlight CV-best row
            tbl[(best_idx + 1, j)].set_facecolor('#d6eaf8')
        ax.text(0.5, 0.02,
                f'Best model by cross-validation: {best_name}  '
                f'(test acc={res.loc[best_idx, "Test_acc"]:.3f}, '
                f'AUC={res.loc[best_idx, "AUC"]:.3f})',
                ha='center', transform=ax.transAxes, fontsize=10, style='italic')
        save(fig, 'results_table')

        # ROC curves (held-out test set)
        fig, ax = plt.subplots(figsize=(7, 7))
        for name, mdl in fitted.items():
            try:
                proba = mdl.predict_proba(Xte)[:, 1]
                fpr, tpr, _ = roc_curve(yte, proba)
                a = roc_auc_score(yte, proba)
                ax.plot(fpr, tpr, lw=1.8, label=f'{name} (AUC={a:.3f})')
            except Exception:
                pass
        ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.6)
        ax.set_xlabel('False positive rate')
        ax.set_ylabel('True positive rate')
        ax.set_title(f'{label}: ROC curves (held-out test set)')
        ax.legend(loc='lower right', fontsize=9)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
        save(fig, 'roc')

        # permutation importance of CV-best model
        best_mdl = fitted[best_name]
        pi = permutation_importance(best_mdl, Xte, yte, n_repeats=30,
                                    random_state=seed, scoring='accuracy',
                                    n_jobs=-1)
        order = np.argsort(pi.importances_mean)[::-1][:20]
        names = [feats[i] for i in order][::-1]
        means = pi.importances_mean[order][::-1]
        errs = pi.importances_std[order][::-1]
        fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(order))))
        ax.barh(range(len(order)), means, xerr=errs, color='#5dade2')
        ax.set_yticks(range(len(order)))
        ax.set_yticklabels(names, fontsize=8)
        ax.set_xlabel('Permutation importance (mean accuracy drop)')
        ax.set_title(f'{label}: top {len(order)} features \u2014 {best_name}')
        save(fig, 'feature_importance')

        # ---- group palette + standardized features for the next two pages ----
        # Exploratory visualization only (not part of the classifier): impute and
        # standardize the full sample so the two groups can be compared on a
        # common scale. ASD = #DA005D, Control/TD = #00473A.
        Xz = StandardScaler().fit_transform(
            SimpleImputer(strategy='median').fit_transform(X))
        asd, ctrl = (y == 1), (y == 0)
        rng = np.random.default_rng(seed)

        # Page 4 -- PCA separation scatter (dots, group-colored)
        pcs = PCA(n_components=2, random_state=seed).fit(Xz)
        proj = pcs.transform(Xz)
        ev = pcs.explained_variance_ratio_ * 100
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.scatter(proj[ctrl, 0], proj[ctrl, 1], s=28, c=CTRL_COLOR,
                   edgecolors='white', linewidths=0.4, alpha=0.8, label='Control')
        ax.scatter(proj[asd, 0], proj[asd, 1], s=28, c=ASD_COLOR,
                   edgecolors='white', linewidths=0.4, alpha=0.8, label='ASD')
        ax.set_xlabel(f'PC1 ({ev[0]:.1f}% var)')
        ax.set_ylabel(f'PC2 ({ev[1]:.1f}% var)')
        ax.set_title(f'{label}: PCA of harmonized features by group')
        ax.legend(loc='best', fontsize=10)
        save(fig, 'pca')

        # Page 5 -- top-feature strip plots (jittered dots, group-colored)
        top = order[:6]
        fig, ax = plt.subplots(figsize=(9, 5.5))
        for i, fidx in enumerate(top):
            for mask, col, off in ((ctrl, CTRL_COLOR, -0.18), (asd, ASD_COLOR, 0.18)):
                xj = i + off + rng.uniform(-0.07, 0.07, size=mask.sum())
                ax.scatter(xj, Xz[mask, fidx], s=16, c=col,
                           edgecolors='white', linewidths=0.3, alpha=0.75)
        ax.set_xticks(range(len(top)))
        ax.set_xticklabels([feats[i] for i in top], rotation=30,
                           ha='right', fontsize=8)
        ax.set_ylabel('Standardized feature value (z)')
        ax.set_title(f'{label}: top discriminative features by group '
                     f'(left = Control, right = ASD)')
        handles = [plt.Line2D([], [], marker='o', ls='', mfc=CTRL_COLOR,
                              mec='white', label='Control'),
                   plt.Line2D([], [], marker='o', ls='', mfc=ASD_COLOR,
                              mec='white', label='ASD')]
        ax.legend(handles=handles, loc='best', fontsize=9)
        save(fig, 'top_features')

    res.to_csv(os.path.join(RESULTS, f'{label}_results.csv'), index=False)
    print(f"  >> CV-best: {best_name}  "
          f"test acc={res.loc[best_idx, 'Test_acc']:.3f}  "
          f"AUC={res.loc[best_idx, 'AUC']:.3f}")
    print(f"  >> saved 5 PDFs to {RESULTS}/ with prefix '{label}_'")
    return res


run_ml('/kaggle/input/datasets/saeedrezaeiafshar/gsr-nogsr-abide-cbt-results/GSR_results/features_ComBat_GSR.csv', 'GSR')
run_ml('/kaggle/input/datasets/saeedrezaeiafshar/gsr-nogsr-abide-cbt-results/noGSR_results/features_ComBat_noGSR.csv', 'no-GSR')
print("\nReport these accuracy/AUC values back for the manuscript.")